# Isolation Forest

Adnan Masood · University of South Florida

Sort transactions from most unusual to least unusual so investigators can choose which to inspect first. An Isolation Forest score measures how easily a row is separated from the sample using its inputs. It is not the chance a transaction is fraudulent and does not establish wrongdoing.

**Dataset:** The computer creates 1,000 fictional background transactions and adds five rows designed to look unusual. Each row has four numbers: amount, hour, activity count, and how much activity differs from that customer’s usual level. Seed 12 makes the generated sample repeatable. The five planted rows are known only because the exercise inserted them; they are not confirmed fraud cases.

**Task:** Fit a forest of 80 isolation trees and sort the rows by unusualness. Count how many of the ten highest-ranked rows are among the five planted examples; this measures recovery in this synthetic exercise, not real fraud detection. Also count how many rows the contamination setting marks for review.

**Compare:** Compare the ten highest anomaly scores with a simple rule that picks the ten largest amounts. Both methods select ten rows, so the comparison asks whether amount alone finds the planted examples as often as the four-feature score. This small artificial check does not predict either method’s real-world fraud yield.

**Runtime:** This notebook uses NumPy and scikit-learn. score_samples returns smaller numbers for more isolated rows, so the code reverses the sign to rank the most unusual first. The contamination setting chooses a threshold for flagging rows; changing it alone changes how many are flagged, not the score order. Check the installed library behavior before applying this cutoff in production.


In [ ]:
import numpy as np
from sklearn.ensemble import IsolationForest
# Use a fixed seed so both rankings use the same fictional transactions.
rng = np.random.default_rng(12)
normal = np.c_[rng.lognormal(3.2,.5,1000),rng.normal(14,3,1000),
    rng.poisson(2,1000),rng.normal(0,1,1000)]
anomaly = np.array([[4500,3,18,8],[2800,2,22,11],[15,4,35,9],
    [9000,13,1,0],[3200,23,16,7]])
X = np.vstack([normal, anomaly])
# Contamination sets a flagging cutoff; score order is checked separately.
iso = IsolationForest(n_estimators=80, contamination=.01, random_state=12)
iso.fit(X)
# sklearn scores lower for isolated rows, so reverse the sign for ranking.
score = -iso.score_samples(X)
# Compare the first ten rows selected by score with the planted examples.
top = np.argsort(score)[-10:][::-1]
print("Top 10 indices:", top)
print("Synthetic anomaly yield@10:", (top >= len(normal)).mean())
# Use the same ten-row capacity for the simple amount-only rule.
rule = np.argsort(X[:,0])[-10:]
print("Amount-rule yield@10:", (rule >= len(normal)).mean())
print("Flagged by contamination cutoff:", int((iso.predict(X) == -1).sum()))


## Try an experiment

Run the initial example and record its results. Then make one of these changes:

### Flag about 5% using contamination

Compare the number flagged by predict. The raw scores and top-ten ranking stay unchanged.

Replace:
```python
contamination=.01
```
With:
```python
contamination=.05
```

### Increase forest size

Compare top-ten indices and planted-anomaly yield with the 80-tree fit.

Replace:
```python
n_estimators=80
```
With:
```python
n_estimators=150
```



In [ ]:
# Run contamination=0.01 and contamination=0.05 with the same seed.
# Compare the top-ten indices, yield, and number flagged by predict.


## Break it

Remove the customer-relative activity column by fitting and scoring only X[:, :3]. Check which of the five planted examples move into or out of the first ten. A small payment can still be unusual for an account, which is why amount alone may miss behavior that differs from a customer’s history.


In [ ]:
# Replace iso.fit(X) with iso.fit(X[:, :3]); use X[:, :3] in score_samples and predict too.
# Compare top-ten indices with the full-feature run and inspect the five planted rows.


## What to submit

- The number of planted examples in the first ten rows for Isolation Forest and the amount-only rule, clearly labeled as synthetic teaching outcomes.
- The count flagged at two contamination settings and evidence that changing the threshold does not reorder the raw scores.
- The result after removing customer-relative activity and a list of evidence an investigator would check before taking action.

## Optional extension

A fraud classifier needs transaction outcomes verified by investigators and evaluation on separate later transactions. Those labels and tests are not supplied by this exercise.
